# Garmin Payload Exploration

Purpose:

- Import raw Garmin running activity files from the S3 FIT landing zone.
- Parse FIT messages into three analytical entities:
  - `sessions`: one row per run, used for run-level metrics.
  - `records`: second-by-second telemetry, used for pace, heart rate, cadence, elevation, and split-style analysis.
  - `events`: activity event messages, used to identify timer boundaries and recovery heart rate.
- Import daily Garmin health JSON payloads from the S3 health landing zone for `hrv`, `rhr`, `sleep`, and `heart_rates`.
- Inspect available fields across FIT entities and health payloads.
- Validate which fields are reliable enough for bronze ingestion and silver candidate extraction.
- Save representative exploration outputs for schema design.

This notebook is exploratory. Reusable Garmin authentication, S3 download, parsing, file IO, and production ingestion logic should live in `ingest/garmin/**`; the notebook should only orchestrate discovery and inspect outputs.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

In [ ]:
import os
import tempfile
from pathlib import Path

import boto3
from botocore.exceptions import BotoCoreError, ClientError, NoCredentialsError
from dotenv import load_dotenv

from ingest.garmin.fit_store import normalize_s3_prefix
from ingest.garmin.health_store import DEFAULT_HEALTH_S3_PREFIX
from ingest.garmin.paths import get_project_root

load_dotenv(get_project_root() / ".env")

s3_staging_context = tempfile.TemporaryDirectory(prefix="garmin-payload-exploration-")
s3_staging_root = Path(s3_staging_context.name)


def s3_uri(bucket: str | None, prefix: str) -> str:
    if not bucket:
        return "s3://<missing-bucket>/" + prefix
    return f"s3://{bucket}/{prefix}" if prefix else f"s3://{bucket}"


def list_s3_keys(bucket: str, prefix: str, suffix: str) -> list[str]:
    client = boto3.client("s3")
    normalized_prefix = normalize_s3_prefix(prefix)
    paginate_args = {"Bucket": bucket}

    if normalized_prefix:
        paginate_args["Prefix"] = f"{normalized_prefix}/"

    keys: list[str] = []
    paginator = client.get_paginator("list_objects_v2")

    for page in paginator.paginate(**paginate_args):
        for item in page.get("Contents", []):
            key = item.get("Key")
            if isinstance(key, str) and key.endswith(suffix):
                keys.append(key)

    return sorted(keys)


def s3_relative_path(key: str, prefix: str) -> Path:
    normalized_prefix = normalize_s3_prefix(prefix)

    if normalized_prefix and key.startswith(f"{normalized_prefix}/"):
        return Path(key.removeprefix(f"{normalized_prefix}/"))

    return Path(key.rsplit("/", maxsplit=1)[-1])


def download_s3_keys(bucket: str, keys: list[str], prefix: str, destination_root: Path) -> list[Path]:
    client = boto3.client("s3")
    downloaded_paths: list[Path] = []

    for key in keys:
        destination_path = destination_root / s3_relative_path(key, prefix)
        destination_path.parent.mkdir(parents=True, exist_ok=True)
        client.download_file(bucket, key, str(destination_path))
        downloaded_paths.append(destination_path)

    return downloaded_paths


def print_s3_access_error(kind: str, error: Exception) -> None:
    print(f"Unable to read {kind} payloads from S3: {type(error).__name__}: {error}")
    print("Confirm AWS credentials and the configured bucket/prefix before rerunning this notebook.")

In [ ]:
from ingest.garmin.fields import EVENT_FIELDS, RECORD_FIELDS, SESSION_FIELDS
from ingest.garmin.parser import parse_fit_files

fit_s3_bucket = os.getenv("GARMIN_FIT_S3_BUCKET")
fit_s3_prefix = normalize_s3_prefix(os.getenv("GARMIN_FIT_S3_PREFIX", "garmin/fit"))
fit_staging_dir = s3_staging_root / "fit"

print(f"FIT S3 source: {s3_uri(fit_s3_bucket, fit_s3_prefix)}")

sessions = pd.DataFrame(columns=[*SESSION_FIELDS, "run_id"])
events = pd.DataFrame(columns=[*EVENT_FIELDS, "run_id"])
records = pd.DataFrame(columns=[*RECORD_FIELDS, "run_id"])
fit_paths: list[Path] = []

if not fit_s3_bucket:
    print("GARMIN_FIT_S3_BUCKET is not set.")
    print("Run:")
    print("    uv run python scripts/download_garmin_fit.py --destination s3 --mode range-overwrite --start-date <date> --end-date <date>")
else:
    try:
        fit_keys = list_s3_keys(fit_s3_bucket, fit_s3_prefix, suffix=".fit")
        print(f"FIT files found in S3: {len(fit_keys)}")
        fit_paths = download_s3_keys(fit_s3_bucket, fit_keys, fit_s3_prefix, fit_staging_dir)
    except (BotoCoreError, ClientError, NoCredentialsError) as exc:
        print_s3_access_error("FIT", exc)

if len(fit_paths) == 0:
    print("No FIT files staged from S3.")
    print("Run:")
    print("    uv run python scripts/download_garmin_fit.py --destination s3 --mode range-overwrite --start-date <date> --end-date <date>")
else:
    print(f"Parsing {len(fit_paths)} staged FIT files from: {fit_staging_dir}")
    parsed = parse_fit_files(fit_paths)
    sessions = parsed["sessions"]
    events = parsed["events"]
    records = parsed["records"]
    print(f"Parsed {len(records)} records from {len(fit_paths)} FIT files.")

In [ ]:
entity_shapes = pd.DataFrame(
    [
        {
            "entity": "sessions",
            "rows": len(sessions),
            "columns": len(sessions.columns),
            "unique_runs": sessions["run_id"].nunique() if "run_id" in sessions.columns else None,
        },
        {
            "entity": "events",
            "rows": len(events),
            "columns": len(events.columns),
            "unique_runs": events["run_id"].nunique() if "run_id" in events.columns else None,
        },
        {
            "entity": "records",
            "rows": len(records),
            "columns": len(records.columns),
            "unique_runs": records["run_id"].nunique() if "run_id" in records.columns else None,
        },
    ]
)

entity_shapes

### Validate field reliability

In [ ]:
def column_inventory(df: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "entity": entity,
            "column": df.columns,
            "dtype": [str(dtype) for dtype in df.dtypes],
            "non_null_count": df.notna().sum().values,
            "null_count": df.isna().sum().values,
            "null_pct": [df[column].isna().mean() * 100 for column in df.columns],
            "unique_count": [df[column].nunique(dropna=True) for column in df.columns],
        }
    ).sort_values(["null_pct", "column"])

field_inventory = pd.concat(
    [
        column_inventory(sessions, "sessions"),
        column_inventory(events, "events"),
        column_inventory(records, "records"),
    ],
    ignore_index=True,
)

# field_inventory

In [ ]:
candidate_fields = {
    "sessions": [
        "run_id",
        "timestamp",
        "start_time",
        "total_elapsed_time",
        "total_timer_time",
        "total_distance",
        "sport",
        "sub_sport",
        "avg_heart_rate",
        "max_heart_rate",
        "enhanced_avg_speed",
        "avg_speed",
        "enhanced_max_speed",
        "max_speed",
        "total_ascent",
        "total_descent",
        "total_calories",
        "total_training_effect",
        "total_anaerobic_training_effect",
        "avg_cadence",
        "max_cadence",
        "avg_running_cadence",
        "max_running_cadence",
    ],
    "events": [
        "run_id",
        "timestamp",
        "event",
        "event_type",
        "data",
    ],
    "records": [
        "run_id",
        "timestamp",
        "distance",
        "heart_rate",
        "enhanced_speed",
        "enhanced_altitude",
        "cadence",
        "temperature",
        "position_lat",
        "position_long",
        "position_lat_deg",
        "position_long_deg",
        "stance_time",
        "vertical_oscillation",
        "vertical_ratio",
        "step_length",
    ],
}

In [ ]:
candidate_inventory = pd.concat(
    [
        field_inventory[
            field_inventory["entity"].eq(entity)
            & field_inventory["column"].isin(fields)
        ]
        for entity, fields in candidate_fields.items()
    ],
    ignore_index=True,
)

In [ ]:
FIELD_CLASSIFICATION_RULES = {
    "required_max_null_pct": 0.0,
    "recommended_max_null_pct": 20.0,
    "optional_max_null_pct": 80.0,
}

def classify_field(null_pct: float) -> str:
    if null_pct == 0:
        return "required_or_stable"
    if null_pct <= FIELD_CLASSIFICATION_RULES["recommended_max_null_pct"]:
        return "recommended"
    if null_pct <= FIELD_CLASSIFICATION_RULES["optional_max_null_pct"]:
        return "optional"
    return "exclude_now"

candidate_inventory = candidate_inventory.copy()
candidate_inventory["reliability_class"] = candidate_inventory["null_pct"].apply(classify_field)

candidate_inventory = candidate_inventory.sort_values(["entity", "reliability_class", "null_pct", "column"]).reset_index(drop=True)
candidate_inventory

### Validate sessions

In [ ]:
session_quality_checks = {
    "session_rows": len(sessions),
    "unique_runs": sessions["run_id"].nunique(),
    "duplicate_run_ids": int(sessions["run_id"].duplicated().sum()),
}

session_quality_checks

In [ ]:
session_preview_columns = [
    column
    for column in [
        "run_id",
        "start_time",
        "timestamp",
        "sport",
        "sub_sport",
        "total_distance",
        "total_timer_time",
        "total_elapsed_time",
        "avg_heart_rate",
        "max_heart_rate",
        "enhanced_avg_speed",
        "avg_speed",
        "total_ascent",
        "total_descent",
    ]
    if column in sessions.columns
]

sessions[session_preview_columns].head()

### Validate records

In [ ]:
records_per_run = (
    records.groupby("run_id")
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

records_per_run.describe()

In [ ]:
record_quality_checks = {
    "record_rows": len(records),
    "unique_runs": records["run_id"].nunique(),
    "timestamp_null_pct": f"{records['timestamp'].isna().mean() * 100:.4f}%" if "timestamp" in records.columns else None,
    "distance_null_pct": f"{records['distance'].isna().mean() * 100:.4f}%" if "distance" in records.columns else None,
    "heart_rate_null_pct": f"{records['heart_rate'].isna().mean() * 100:.4f}%" if "heart_rate" in records.columns else None,
    "enhanced_speed_null_pct": f"{records['enhanced_speed'].isna().mean() * 100:.4f}%" if "enhanced_speed" in records.columns else None,
}

record_quality_checks

### Validate events and recovery HR

In [ ]:
event_counts = (
    events.groupby(["event", "event_type"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

event_counts

In [ ]:
recovery_events = events[
    events["event"].astype(str).str.contains("recovery_hr", case=False, na=False)
].copy().reset_index(drop=True)

if "data" in recovery_events.columns:
    recovery_events["recovery_hr"] = pd.to_numeric(
        recovery_events["data"],
        errors="coerce",
    )

recovery_events.head()

In [ ]:
recovery_event_count_distribution = (
    recovery_events.groupby("run_id")
    .size()
    .value_counts()
    .rename_axis("recovery_event_count")
    .reset_index(name="run_count")
    .sort_values("recovery_event_count")
    .reset_index(drop=True)
)

recovery_event_count_distribution

In [ ]:
recovery_coverage = {
    "runs_with_recovery_events": recovery_events["run_id"].nunique(),
    "recovery_coverage_pct": f"{(recovery_events['run_id'].nunique() / sessions['run_id'].nunique() * 100):.2f}%"
    if sessions["run_id"].nunique() > 0
    else None,
}

recovery_coverage

In [ ]:
last_record_per_run = (
    records.sort_values(["run_id", "timestamp"])
    .groupby("run_id", as_index=False)
    .tail(1)
)

last_record_columns = [
    column
    for column in ["run_id", "timestamp", "heart_rate", "distance"]
    if column in last_record_per_run.columns
]

last_records = last_record_per_run[last_record_columns].rename(
    columns={
        "timestamp": "last_record_timestamp",
        "heart_rate": "final_record_heart_rate",
        "distance": "final_record_distance",
    }
)

last_records.head()

In [ ]:
if recovery_events.empty:
    print("No recovery events found. Cannot validate recovery HR drop.")
elif "recovery_hr" not in recovery_events.columns:
    print("recovery_events exists, but `recovery_hr` has not been derived yet.")
    display(recovery_events.head(20))
else:
    recovery_validation = recovery_events.merge(last_records, on="run_id", how="left")

    if "final_record_heart_rate" in recovery_validation.columns:
        recovery_validation["final_hr_minus_recovery_hr"] = (
            recovery_validation["final_record_heart_rate"]
            - recovery_validation["recovery_hr"]
        )

    display(
        recovery_validation[
            [
                column
                for column in [
                    "run_id",
                    "last_record_timestamp",
                    "recovery_hr",
                    "final_record_heart_rate",
                    "final_hr_minus_recovery_hr",
                    "final_record_distance",
                ]
                if column in recovery_validation.columns
            ]
        ].head()
    )

### Save representative exploration outputs

In [ ]:
from ingest.garmin.paths import get_garmin_exploration_dir

exploration_dir = get_garmin_exploration_dir()
print(exploration_dir)

#### field inventory

In [ ]:
field_inventory_path = exploration_dir / "fit_field_inventory.csv"
candidate_inventory_path = exploration_dir / "fit_candidate_field_inventory.csv"

field_inventory.to_csv(field_inventory_path, index=False)
candidate_inventory.to_csv(candidate_inventory_path, index=False)

print(f"Field inventory saved to: {field_inventory_path}")
print(f"Candidate field inventory saved to: {candidate_inventory_path}")

#### representative samples

In [ ]:
sessions_sample_path = exploration_dir / "sessions.sample.csv"
events_sample_path = exploration_dir / "events.sample.csv"
records_sample_path = exploration_dir / "records.sample.csv"

sessions.head(100).to_csv(sessions_sample_path, index=False)
events.head(500).to_csv(events_sample_path, index=False)
records.head(5000).to_csv(records_sample_path, index=False)

print(f"Sessions sample dataset saved to: {sessions_sample_path}")
print(f"Events sample dataset saved to: {events_sample_path}")
print(f"Records sample dataset saved to: {records_sample_path}")

#### recovery HR sample

In [ ]:
recovery_events_path = exploration_dir / "recovery_events.sample.csv"

recovery_events.to_csv(recovery_events_path, index=False)

print(f"Recovery events dataset saved to: {recovery_events_path}")

#### schema recommendation draft

In [ ]:
schema_recommendation = candidate_inventory[
    [
        "entity",
        "column",
        "dtype",
        "non_null_count",
        "null_pct",
        "unique_count",
        "reliability_class",
    ]
].sort_values(["entity", "reliability_class", "null_pct", "column"])

schema_recommendation_path = exploration_dir / "fit_bronze_schema_recommendation.csv"

schema_recommendation.to_csv(schema_recommendation_path, index=False)

print(f"Schema recommendation saved to: {schema_recommendation_path}")

#### run coverage summary

In [ ]:
coverage_summary = pd.DataFrame(
    [
        {
            "metric": "session_rows",
            "value": len(sessions),
        },
        {
            "metric": "session_unique_runs",
            "value": sessions["run_id"].nunique(),
        },
        {
            "metric": "event_rows",
            "value": len(events),
        },
        {
            "metric": "event_unique_runs",
            "value": events["run_id"].nunique(),
        },
        {
            "metric": "record_rows",
            "value": len(records),
        },
        {
            "metric": "record_unique_runs",
            "value": records["run_id"].nunique(),
        },
        {
            "metric": "recovery_event_rows",
            "value": len(recovery_events),
        },
        {
            "metric": "recovery_event_unique_runs",
            "value": recovery_events["run_id"].nunique(),
        },
    ]
)

coverage_summary_path = exploration_dir / "fit_coverage_summary.csv"

coverage_summary.to_csv(coverage_summary_path, index=False)

print(f"Coverage summary saved to: {coverage_summary_path}")

## Explore daily health JSON payloads

Daily health payloads are expected under `s3://<bucket>/garmin/health/daily/calendar_date=YYYY-MM-DD/{payload_type}.json`, where `payload_type` is one of `hrv`, `rhr`, `sleep`, or `heart_rates`. The notebook stages matching S3 objects into a temporary local directory before using the existing bronze-shaping helper.

In [ ]:
from ingest.garmin.bronze_schema import HEALTH_PAYLOAD_TABLE
from ingest.garmin.health_bronze import build_health_bronze_frame, discover_health_payload_files
from ingest.garmin.health_store import HEALTH_PAYLOAD_TYPES

health_s3_bucket = os.getenv("GARMIN_HEALTH_S3_BUCKET") or os.getenv("GARMIN_FIT_S3_BUCKET")
health_s3_prefix = normalize_s3_prefix(os.getenv("GARMIN_HEALTH_S3_PREFIX", DEFAULT_HEALTH_S3_PREFIX))
health_staging_dir = s3_staging_root / "health" / "daily"

print(f"Health S3 source: {s3_uri(health_s3_bucket, health_s3_prefix)}")

health_source_files = []
health_payloads = pd.DataFrame(columns=HEALTH_PAYLOAD_TABLE.columns)

if not health_s3_bucket:
    print("GARMIN_HEALTH_S3_BUCKET or GARMIN_FIT_S3_BUCKET is not set.")
    print("Run:")
    print("    uv run python scripts/download_garmin_health.py --destination s3 --start-date <date> --end-date <date>")
else:
    try:
        health_keys = list_s3_keys(health_s3_bucket, health_s3_prefix, suffix=".json")
        print(f"Health payload files found in S3: {len(health_keys)}")
        download_s3_keys(health_s3_bucket, health_keys, health_s3_prefix, health_staging_dir)
        health_source_files = discover_health_payload_files(health_staging_dir)
    except (BotoCoreError, ClientError, NoCredentialsError) as exc:
        print_s3_access_error("health", exc)

if len(health_source_files) == 0:
    print("No health payload files staged from S3.")
    print("Run:")
    print("    uv run python scripts/download_garmin_health.py --destination s3 --start-date <date> --end-date <date>")
else:
    health_payloads = build_health_bronze_frame(health_source_files)
    print(f"Built {len(health_payloads)} bronze-shaped health payload rows from staged S3 files.")

In [ ]:
observed_health_payload_types = sorted(health_payloads["payload_type"].dropna().astype(str).unique())

health_entity_shapes = pd.DataFrame(
    [
        {
            "entity": "health_payloads",
            "payload_type": "all",
            "rows": len(health_payloads),
            "columns": len(health_payloads.columns),
            "unique_dates": health_payloads["calendar_date"].nunique(),
            "payload_types_observed": ", ".join(observed_health_payload_types),
        },
        *[
            {
                "entity": "health_payloads",
                "payload_type": payload_type,
                "rows": int((health_payloads["payload_type"] == payload_type).sum()),
                "columns": len(health_payloads.columns),
                "unique_dates": health_payloads.loc[
                    health_payloads["payload_type"] == payload_type,
                    "calendar_date",
                ].nunique(),
                "payload_types_observed": payload_type if payload_type in observed_health_payload_types else "",
            }
            for payload_type in HEALTH_PAYLOAD_TYPES
        ],
    ]
)

health_entity_shapes

In [ ]:
health_payload_inventory = column_inventory(health_payloads, "health_payloads")
health_payload_inventory

In [ ]:
expected_health_payload_types = pd.DataFrame({"payload_type": list(HEALTH_PAYLOAD_TYPES)})
observed_health_payload_counts = (
    health_payloads.groupby("payload_type", dropna=False)
    .size()
    .reset_index(name="row_count")
)

health_payload_type_counts = (
    expected_health_payload_types.merge(
        observed_health_payload_counts,
        on="payload_type",
        how="left",
    )
    .fillna({"row_count": 0})
)
health_payload_type_counts["row_count"] = health_payload_type_counts["row_count"].astype(int)
health_payload_type_counts

In [ ]:
if health_payloads.empty:
    health_date_payload_matrix = pd.DataFrame(columns=["calendar_date", *HEALTH_PAYLOAD_TYPES])
else:
    health_payload_presence = (
        health_payloads[["calendar_date", "payload_type"]]
        .drop_duplicates()
        .assign(present=1)
    )
    health_date_payload_matrix = (
        health_payload_presence.pivot_table(
            index="calendar_date",
            columns="payload_type",
            values="present",
            aggfunc="max",
            fill_value=0,
        )
        .reset_index()
        .rename_axis(columns=None)
    )

    for payload_type in HEALTH_PAYLOAD_TYPES:
        if payload_type not in health_date_payload_matrix.columns:
            health_date_payload_matrix[payload_type] = 0

    health_date_payload_matrix = health_date_payload_matrix[["calendar_date", *HEALTH_PAYLOAD_TYPES]].sort_values("calendar_date")

health_date_payload_matrix

### Inspect nested health payload keys

In [ ]:
import json
from collections import Counter
from typing import Any


def parse_raw_payload(value: object) -> Any:
    if value is None:
        return None
    if isinstance(value, (dict, list)):
        return value
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return None


def iter_leaf_paths(value: Any, prefix: str = "$"):
    if isinstance(value, dict):
        if not value:
            yield prefix
            return
        for key, child in value.items():
            yield from iter_leaf_paths(child, f"{prefix}.{key}")
        return

    if isinstance(value, list):
        if not value:
            yield f"{prefix}[]"
            return
        for item in value:
            yield from iter_leaf_paths(item, f"{prefix}[]")
        return

    yield prefix


health_payload_row_counts = health_payloads.groupby("payload_type").size().to_dict()
health_payload_key_counts: Counter[tuple[str, str]] = Counter()

for _, row in health_payloads.iterrows():
    payload = parse_raw_payload(row.get("raw_payload"))
    if payload is None:
        continue

    payload_type = str(row["payload_type"])
    for json_path in set(iter_leaf_paths(payload)):
        health_payload_key_counts[(payload_type, json_path)] += 1

health_payload_key_inventory = pd.DataFrame(
    [
        {
            "payload_type": payload_type,
            "json_path": json_path,
            "payload_rows_with_key": row_count,
            "payload_row_count": int(health_payload_row_counts.get(payload_type, 0)),
            "coverage_pct": row_count / health_payload_row_counts[payload_type] * 100
            if health_payload_row_counts.get(payload_type, 0) > 0
            else None,
        }
        for (payload_type, json_path), row_count in sorted(health_payload_key_counts.items())
    ],
    columns=[
        "payload_type",
        "json_path",
        "payload_rows_with_key",
        "payload_row_count",
        "coverage_pct",
    ],
)

health_payload_key_inventory

### Validate health metric candidate paths

Candidate extraction mirrors the current `health_days` dbt paths. This cell is exploratory only; production parsing stays in dbt silver models.

In [ ]:
HEALTH_METRIC_CANDIDATE_COLUMNS = [
    "calendar_date",
    "payload_type",
    "hrv_summary_last_night_avg",
    "hrv_summary_weekly_avg",
    "hrv_last_night_avg",
    "hrv_value_raw",
    "hrv_value_candidate",
    "hrv_summary_status",
    "hrv_status_raw",
    "hrv_status_candidate",
    "rhr_resting_heart_rate",
    "rhr_metrics_map_resting_heart_rate",
    "rhr_value_raw",
    "rhr_resting_heart_rate_candidate",
    "sleep_score_daily_sleep_scores_overall",
    "sleep_score_daily_sleep_dto",
    "sleep_score_overall",
    "sleep_score_raw",
    "sleep_score_candidate",
    "sleep_time_seconds_daily_sleep_dto",
    "sleep_time_seconds_raw",
    "sleep_duration_seconds_raw",
    "sleep_duration_seconds_candidate",
    "sleep_start_timestamp_gmt_daily_sleep_dto",
    "sleep_start_timestamp_gmt_raw",
    "sleep_start_time_gmt_raw",
    "sleep_start_time_raw",
    "sleep_end_timestamp_gmt_daily_sleep_dto",
    "sleep_end_timestamp_gmt_raw",
    "sleep_end_time_gmt_raw",
    "sleep_end_time_raw",
    "heart_rates_resting_heart_rate",
    "heart_rates_resting_heart_rate_candidate",
]


def get_nested_value(payload: Any, path: str) -> Any:
    current = payload
    for part in path.split("."):
        if not isinstance(current, dict):
            return None
        current = current.get(part)
        if current is None:
            return None
    return current


def first_present(payload: Any, paths: list[str]) -> Any:
    for path in paths:
        value = get_nested_value(payload, path)
        if value is not None and value != "":
            return value
    return None


def health_metric_candidate_row(row: pd.Series) -> dict[str, Any]:
    payload = parse_raw_payload(row.get("raw_payload"))
    payload_type = str(row.get("payload_type"))
    candidate = {column: None for column in HEALTH_METRIC_CANDIDATE_COLUMNS}
    candidate["calendar_date"] = row.get("calendar_date")
    candidate["payload_type"] = payload_type

    if payload is None:
        return candidate

    if payload_type == "hrv":
        candidate["hrv_summary_last_night_avg"] = get_nested_value(payload, "hrvSummary.lastNightAvg")
        candidate["hrv_summary_weekly_avg"] = get_nested_value(payload, "hrvSummary.weeklyAvg")
        candidate["hrv_last_night_avg"] = get_nested_value(payload, "lastNightAvg")
        candidate["hrv_value_raw"] = get_nested_value(payload, "hrvValue")
        candidate["hrv_value_candidate"] = first_present(
            payload,
            [
                "hrvSummary.lastNightAvg",
                "hrvSummary.weeklyAvg",
                "lastNightAvg",
                "hrvValue",
            ],
        )
        candidate["hrv_summary_status"] = get_nested_value(payload, "hrvSummary.status")
        candidate["hrv_status_raw"] = get_nested_value(payload, "status")
        candidate["hrv_status_candidate"] = first_present(
            payload,
            ["hrvSummary.status", "status"],
        )

    if payload_type == "rhr":
        candidate["rhr_resting_heart_rate"] = get_nested_value(payload, "restingHeartRate")
        candidate["rhr_metrics_map_resting_heart_rate"] = get_nested_value(
            payload,
            "allMetrics.metricsMap.WELLNESS_RESTING_HEART_RATE",
        )
        candidate["rhr_value_raw"] = get_nested_value(payload, "value")
        candidate["rhr_resting_heart_rate_candidate"] = first_present(
            payload,
            [
                "restingHeartRate",
                "allMetrics.metricsMap.WELLNESS_RESTING_HEART_RATE",
                "value",
            ],
        )

    if payload_type == "sleep":
        candidate["sleep_score_daily_sleep_scores_overall"] = get_nested_value(
            payload,
            "dailySleepDTO.sleepScores.overall.value",
        )
        candidate["sleep_score_daily_sleep_dto"] = get_nested_value(payload, "dailySleepDTO.sleepScore")
        candidate["sleep_score_overall"] = get_nested_value(payload, "overallSleepScore.value")
        candidate["sleep_score_raw"] = get_nested_value(payload, "sleepScore")
        candidate["sleep_score_candidate"] = first_present(
            payload,
            [
                "dailySleepDTO.sleepScores.overall.value",
                "dailySleepDTO.sleepScore",
                "overallSleepScore.value",
                "sleepScore",
            ],
        )
        candidate["sleep_time_seconds_daily_sleep_dto"] = get_nested_value(payload, "dailySleepDTO.sleepTimeSeconds")
        candidate["sleep_time_seconds_raw"] = get_nested_value(payload, "sleepTimeSeconds")
        candidate["sleep_duration_seconds_raw"] = get_nested_value(payload, "durationInSeconds")
        candidate["sleep_duration_seconds_candidate"] = first_present(
            payload,
            [
                "dailySleepDTO.sleepTimeSeconds",
                "sleepTimeSeconds",
                "durationInSeconds",
            ],
        )
        candidate["sleep_start_timestamp_gmt_daily_sleep_dto"] = get_nested_value(
            payload,
            "dailySleepDTO.sleepStartTimestampGMT",
        )
        candidate["sleep_start_timestamp_gmt_raw"] = get_nested_value(payload, "sleepStartTimestampGMT")
        candidate["sleep_start_time_gmt_raw"] = get_nested_value(payload, "sleepStartTimeGMT")
        candidate["sleep_start_time_raw"] = first_present(
            payload,
            [
                "dailySleepDTO.sleepStartTimestampGMT",
                "sleepStartTimestampGMT",
                "sleepStartTimeGMT",
            ],
        )
        candidate["sleep_end_timestamp_gmt_daily_sleep_dto"] = get_nested_value(
            payload,
            "dailySleepDTO.sleepEndTimestampGMT",
        )
        candidate["sleep_end_timestamp_gmt_raw"] = get_nested_value(payload, "sleepEndTimestampGMT")
        candidate["sleep_end_time_gmt_raw"] = get_nested_value(payload, "sleepEndTimeGMT")
        candidate["sleep_end_time_raw"] = first_present(
            payload,
            [
                "dailySleepDTO.sleepEndTimestampGMT",
                "sleepEndTimestampGMT",
                "sleepEndTimeGMT",
            ],
        )

    if payload_type == "heart_rates":
        candidate["heart_rates_resting_heart_rate"] = get_nested_value(payload, "restingHeartRate")
        candidate["heart_rates_resting_heart_rate_candidate"] = get_nested_value(payload, "restingHeartRate")

    return candidate


health_metric_candidates = pd.DataFrame(
    [health_metric_candidate_row(row) for _, row in health_payloads.iterrows()],
    columns=HEALTH_METRIC_CANDIDATE_COLUMNS,
)

health_metric_candidates

### Health payload quality checks

In [ ]:
health_duplicate_payload_keys = health_payloads[
    health_payloads[["calendar_date", "payload_type"]].duplicated(keep=False)
].sort_values(["calendar_date", "payload_type"])

if health_date_payload_matrix.empty:
    health_missing_expected_payloads = pd.DataFrame(
        columns=["calendar_date", "missing_payload_types", "missing_payload_type_count"]
    )
else:
    health_missing_expected_payloads = health_date_payload_matrix.copy()
    health_missing_expected_payloads["missing_payload_types"] = health_missing_expected_payloads.apply(
        lambda row: ", ".join(
            [payload_type for payload_type in HEALTH_PAYLOAD_TYPES if int(row[payload_type]) == 0]
        ),
        axis=1,
    )
    health_missing_expected_payloads["missing_payload_type_count"] = health_missing_expected_payloads[
        "missing_payload_types"
    ].apply(lambda value: 0 if value == "" else len(value.split(", ")))
    health_missing_expected_payloads = health_missing_expected_payloads[
        ["calendar_date", "missing_payload_types", "missing_payload_type_count"]
    ]
    health_missing_expected_payloads = health_missing_expected_payloads[
        health_missing_expected_payloads["missing_payload_type_count"] > 0
    ].reset_index(drop=True)

health_observed_date_count = health_payloads["calendar_date"].nunique()
health_dates_with_payload = (
    health_payloads[["calendar_date", "payload_type"]]
    .drop_duplicates()
    .groupby("payload_type")
    .size()
    .reset_index(name="dates_with_payload")
)
health_endpoint_coverage = (
    expected_health_payload_types.merge(
        health_dates_with_payload,
        on="payload_type",
        how="left",
    )
    .fillna({"dates_with_payload": 0})
)
health_endpoint_coverage["dates_with_payload"] = health_endpoint_coverage["dates_with_payload"].astype(int)
health_endpoint_coverage["observed_date_count"] = health_observed_date_count
health_endpoint_coverage["coverage_pct"] = health_endpoint_coverage.apply(
    lambda row: row["dates_with_payload"] / row["observed_date_count"] * 100
    if row["observed_date_count"] > 0
    else None,
    axis=1,
)
health_endpoint_coverage = health_endpoint_coverage[
    ["payload_type", "dates_with_payload", "observed_date_count", "coverage_pct"]
]

health_candidate_coverage_specs = {
    "hrv_value_candidate": "hrv",
    "hrv_status_candidate": "hrv",
    "rhr_resting_heart_rate_candidate": "rhr",
    "sleep_score_candidate": "sleep",
    "sleep_duration_seconds_candidate": "sleep",
    "sleep_start_time_raw": "sleep",
    "sleep_end_time_raw": "sleep",
    "heart_rates_resting_heart_rate_candidate": "heart_rates",
}

health_candidate_field_coverage = pd.DataFrame(
    [
        {
            "payload_type": payload_type,
            "candidate_field": column,
            "applicable_row_count": int((health_metric_candidates["payload_type"] == payload_type).sum()),
            "non_null_count": int(
                health_metric_candidates.loc[
                    health_metric_candidates["payload_type"] == payload_type,
                    column,
                ].notna().sum()
            ),
        }
        for column, payload_type in health_candidate_coverage_specs.items()
    ]
)
health_candidate_field_coverage["coverage_pct"] = health_candidate_field_coverage.apply(
    lambda row: row["non_null_count"] / row["applicable_row_count"] * 100
    if row["applicable_row_count"] > 0
    else None,
    axis=1,
)

health_payload_coverage_summary = pd.concat(
    [
        pd.DataFrame(
            [
                {
                    "check_type": "summary",
                    "payload_type": None,
                    "metric": "payload_rows",
                    "numerator": len(health_payloads),
                    "denominator": None,
                    "pct": None,
                },
                {
                    "check_type": "summary",
                    "payload_type": None,
                    "metric": "calendar_dates",
                    "numerator": health_observed_date_count,
                    "denominator": None,
                    "pct": None,
                },
                {
                    "check_type": "quality",
                    "payload_type": None,
                    "metric": "duplicate_calendar_date_payload_type_rows",
                    "numerator": len(health_duplicate_payload_keys),
                    "denominator": len(health_payloads),
                    "pct": len(health_duplicate_payload_keys) / len(health_payloads) * 100
                    if len(health_payloads) > 0
                    else None,
                },
            ]
        ),
        health_endpoint_coverage.assign(
            check_type="endpoint_coverage",
            metric="dates_with_payload",
        ).rename(
            columns={
                "dates_with_payload": "numerator",
                "observed_date_count": "denominator",
                "coverage_pct": "pct",
            }
        )[["check_type", "payload_type", "metric", "numerator", "denominator", "pct"]],
        health_candidate_field_coverage.assign(
            check_type="candidate_field_coverage",
            metric=health_candidate_field_coverage["candidate_field"],
        ).rename(
            columns={
                "non_null_count": "numerator",
                "applicable_row_count": "denominator",
                "coverage_pct": "pct",
            }
        )[["check_type", "payload_type", "metric", "numerator", "denominator", "pct"]],
    ],
    ignore_index=True,
)

health_quality_checks = {
    "duplicate_calendar_date_payload_type_rows": len(health_duplicate_payload_keys),
    "dates_missing_expected_payloads": len(health_missing_expected_payloads),
    "observed_date_count": health_observed_date_count,
}

health_quality_checks

In [ ]:
health_endpoint_coverage

In [ ]:
health_candidate_field_coverage

### Save health exploration outputs

In [ ]:
health_payload_inventory_path = exploration_dir / "health_payload_inventory.csv"
health_payload_key_inventory_path = exploration_dir / "health_payload_key_inventory.csv"
health_metric_candidates_path = exploration_dir / "health_metric_candidates.csv"
health_payload_coverage_summary_path = exploration_dir / "health_payload_coverage_summary.csv"
health_payloads_sample_path = exploration_dir / "health_payloads.sample.csv"

health_payload_inventory.to_csv(health_payload_inventory_path, index=False)
health_payload_key_inventory.to_csv(health_payload_key_inventory_path, index=False)
health_metric_candidates.to_csv(health_metric_candidates_path, index=False)
health_payload_coverage_summary.to_csv(health_payload_coverage_summary_path, index=False)
health_payloads.head(500).to_csv(health_payloads_sample_path, index=False)

print(f"Health payload inventory saved to: {health_payload_inventory_path}")
print(f"Health payload key inventory saved to: {health_payload_key_inventory_path}")
print(f"Health metric candidates saved to: {health_metric_candidates_path}")
print(f"Health payload coverage summary saved to: {health_payload_coverage_summary_path}")
print(f"Health payload sample saved to: {health_payloads_sample_path}")

## Exploration conclusion

This notebook confirms that Garmin payloads can be shaped into the first bronze entities used by the project.

FIT files parse into three useful analytical entities:

- `sessions`: run-level facts suitable for consistency, volume, and summary fitness metrics.
- `records`: high-frequency telemetry suitable for pace/heart-rate analysis, splits, and within-run patterns.
- `events`: event messages suitable for timer state changes and recovery heart rate extraction.

Daily health JSON files can be inventoried as one bronze row per `calendar_date` and `payload_type` for:

- `hrv`
- `rhr`
- `sleep`
- `heart_rates`

The production bronze outputs are:

- `bronze.garmin_fit_sessions`
- `bronze.garmin_fit_records`
- `bronze.garmin_fit_events`
- `bronze.garmin_health_daily_payloads`

Bronze ingestion should preserve source-level values with minimal transformation. Metric derivation such as pace, HR-band efficiency, recovery HR drop, resting heart rate, HRV, and sleep metrics should happen in silver or gold models.